# OSR 508GB — Stage-1 v3 Identity Lock → Stage-2 Preflight

This notebook **does not run the full Stage-2 scan**. It only:
1. mounts Google Drive;
2. builds the canonical Stage-1 v3 identity manifest over 1,788 Parquet shards;
3. verifies the manifest root and 5 real schema variants;
4. performs raw-row SHA round-trip canaries;
5. writes `STAGE2_PREFLIGHT_VERIFIED.json`.

Only after all cells pass should the Stage-2 Direct Miner v3.1 be run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, subprocess, sys

STAGE1_URL = 'https://raw.githubusercontent.com/rayzh2012/Open-Source-Research-OSR-/606cbdef99de7d806775fb9d3da084ed6a5a9c49/tools/osr_stage1_identity_lock_v3.py'
stage1 = Path('/content/osr_stage1_identity_lock_v3.py')
data = urlopen(STAGE1_URL).read()
stage1.write_bytes(data)
print('Stage-1 engine SHA256:', hashlib.sha256(data).hexdigest())
subprocess.run([sys.executable, str(stage1)], check=True)

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, subprocess, sys

PREFLIGHT_URL = 'https://raw.githubusercontent.com/rayzh2012/Open-Source-Research-OSR-/877b276496a707864ef8c13036af2f4959fcba58/tools/osr_stage2_preflight_v1.py'
preflight = Path('/content/osr_stage2_preflight_v1.py')
data = urlopen(PREFLIGHT_URL).read()
preflight.write_bytes(data)
print('Preflight engine SHA256:', hashlib.sha256(data).hexdigest())
subprocess.run([sys.executable, str(preflight)], check=True)

In [ ]:
import json
from pathlib import Path

stage1_dir = Path('/content/drive/MyDrive/OSR_WORK_SPACE/Stage1_Identity_v3')
stage1_verified = json.loads((stage1_dir / 'STAGE1_V3_VERIFIED.json').read_text('utf-8'))
preflight_path = Path('/content/drive/MyDrive/OSR_WORK_SPACE/Stage2_Preflight_v1/STAGE2_PREFLIGHT_VERIFIED.json')
preflight_verified = json.loads(preflight_path.read_text('utf-8'))

assert stage1_verified['verified'] is True
assert stage1_verified['canonical_shards'] == 1788
assert stage1_verified['schema_fingerprint_count'] == 5
assert preflight_verified['verified'] is True
assert preflight_verified['manifest_sha256'] == stage1_verified['manifest_sha256']

print('✅ STAGE-1 V3 VERIFIED')
print('manifest_sha256:', stage1_verified['manifest_sha256'])
print('✅ STAGE-2 PREFLIGHT VERIFIED')
print('🚦 STAGE-2 FULL SCAN IS NOW ALLOWED — this notebook intentionally stops here.')